# Grad-CAM Explanations

This notebook employs GradCAM to see which aspects of the image influence the classifier. We want to see where the model's "attention" is.

In [ ]:
from pathlib import Path
import sys

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data/oct_c8/RetinalOCT_Dataset/RetinalOCT_Dataset"
DATA_DIR = PROJECT_ROOT / "data/RetinalOCT_Dataset"

MODEL_PATH = PROJECT_ROOT / "results/models/resnet50_oct_c8_layer4_finetuned.pt"
print(DATA_DIR.exists())
print(MODEL_PATH.exists())

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

device

In [ ]:
# Load Model
from src.models import build_resnet50
from src.dataset import get_oct_c8_dataloaders

_, _, test_loader, class_names = get_oct_c8_dataloaders(
    DATA_DIR,
    batch_size=1,
    num_workers=0
)

model = build_resnet50(
    num_classes=len(class_names),
    pretrained=False,
    freeze_backbone=False
)

model.load_state_dict(
    torch.load(MODEL_PATH, map_location=device)
)

model = model.to(device)
model.eval()

class_names

In [ ]:
from src.gradcam import GradCAM

target_layer = model.layer4[-1]

gradcam = GradCAM(
    model=model,
    target_layer=target_layer
)

In [ ]:
images, labels = next(iter(test_loader))

images = images.to(device)
labels = labels.to(device)

cam, output = gradcam.generate(images)

probs = F.softmax(output, dim=1)
pred_idx = probs.argmax(dim=1).item()
true_idx = labels.item()

print("True:", class_names[true_idx])
print("Pred:", class_names[pred_idx])
print("Confidence:", probs[0, pred_idx].item())

In [ ]:
img = images[0].detach().cpu()

# unnormalize ImageNet normalization
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

img_vis = img * std + mean
img_vis = img_vis.clamp(0, 1)
img_vis = img_vis.permute(1, 2, 0).numpy()

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(img_vis, cmap="gray")
plt.title(f"True: {class_names[true_idx]}")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img_vis, cmap="gray")
plt.imshow(cam, cmap="jet", alpha=0.45)
plt.title(f"Pred: {class_names[pred_idx]} | p={probs[0, pred_idx].item():.2f}")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def show_gradcam_examples(target_class, n=4):
    target_idx = class_names.index(target_class)

    shown = 0

    for images, labels in test_loader:
        if labels.item() != target_idx:
            continue

        images = images.to(device)
        labels = labels.to(device)

        cam, output = gradcam.generate(images)

        probs = F.softmax(output, dim=1)
        pred_idx = probs.argmax(dim=1).item()

        img = images[0].detach().cpu()

        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

        img_vis = img * std + mean
        img_vis = img_vis.clamp(0, 1)
        img_vis = img_vis.permute(1, 2, 0).numpy()

        plt.figure(figsize=(10, 4))

        plt.subplot(1, 2, 1)
        plt.imshow(img_vis, cmap="gray")
        plt.title(f"True: {target_class}")
        plt.axis("off")

        plt.subplot(1, 2, 2)
        plt.imshow(img_vis, cmap="gray")
        plt.imshow(cam, cmap="jet", alpha=0.45)
        plt.title(
            f"Pred: {class_names[pred_idx]} | "
            f"p={probs[0, pred_idx].item():.2f}"
        )
        plt.axis("off")

        plt.tight_layout()
        plt.show()

        shown += 1

        if shown >= n:
            break

In [ ]:
show_gradcam_examples("AMD", n=5)

In [ ]:
show_gradcam_examples("DRUSEN", n=5)

In [ ]:
show_gradcam_examples("NORMAL", n=5)

Grad-CAM demonstrates that the CNN attends to clinically meaningful retinal structures and pathology-associated regions rather than relying solely on image acquisition artifacts.

The AMD Grad-CAMs now look more coherent than the frozen model. They tend to concentrate around the posterior retina / RPE-choroid region rather than random corners or image edges.

For the next phase of the project, it would be helpful to obtain:

* OCT volume scans for patients with suspected barcoding/hypertransmission.
* OCT volume scans for patients without barcoding/hypertransmission to serve as comparison cases.
* Any existing hypertransmission labels, manual markings, or clinical assessments indicating whether a scan is considered barcode-positive.

For future analyses, it would also be useful to have:

* Longitudinal follow-up information, including progression status, progression rate, or time to progression.
* OCT device information and scan acquisition parameters.
* Physical scale information or metadata needed for pixel-to-micron conversion.
* Information on whether the OCT software already provides retinal layer segmentations, particularly the RPE, Bruch’s membrane (BM), or choroid-sclera interface (CSI).

At this stage, volume scans and positive/negative examples are probably the highest priority. Everything else would be extremely helpful, but those are the pieces needed to get the next phase off the ground.